In [453]:
import pandas as pd
train_csv = pd.read_csv('train.csv')
test_csv = pd.read_csv('test.csv')

display(train_csv)

,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival
...,...,...
9998,You provide support in what countries?,country_support
9999,What countries are you supporting?,country_support
10000,What countries are getting support?,country_support
10001,Are cards available in the EU?,country_support


In [454]:
train_csv['text'] = train_csv['text'].str.lower()
test_csv['text'] = test_csv['text'].str.lower()
print(train_csv['text'][16])

i don't have my card in 1 week.  should i be worried?


In [455]:
import re
train_csv['text'] = train_csv['text'].apply(lambda x: re.sub(r'[^a-z\' ]', '', x))
test_csv['text'] = test_csv['text'].apply(lambda x: re.sub(r'[^a-z\' ]', '', x))
print(train_csv['text'][16])

i don't have my card in  week  should i be worried


In [456]:
train_csv['text'] = train_csv['text'].str.split()
test_csv['text'] = test_csv['text'].str.split()
print(train_csv['text'][16])

['i', "don't", 'have', 'my', 'card', 'in', 'week', 'should', 'i', 'be', 'worried']


In [457]:
import nltk
from nltk.corpus import stopwords

stop_words_en = set(stopwords.words('english'))
train_csv['text'] = train_csv['text'].apply(lambda x: [word for word in x if word not in stop_words_en])
test_csv['text'] = test_csv['text'].apply(lambda x: [word for word in x if word not in stop_words_en])
print(train_csv['text'][16])

['card', 'week', 'worried']


In [458]:
print(train_csv['category'].tolist()[16])

card_arrival


In [459]:
from torchtext.vocab import build_vocab_from_iterator, vocab
text_vocab = build_vocab_from_iterator(train_csv['text'], min_freq=5, specials=["<unk>", "<pad>"]) 
label_vocab = build_vocab_from_iterator([train_csv['category']])
text_vocab.set_default_index(text_vocab["<unk>"])
# label_vocab.set_default_index(label_vocab["<unk>"])

UNK_IDX = text_vocab['<unk>']
PAD_IDX = text_vocab['<pad>']
print(UNK_IDX, PAD_IDX)

INPUT_DIM = len(text_vocab)
OUTPUT_DIM = len(label_vocab)
print(INPUT_DIM, OUTPUT_DIM)

0 1
837 77


In [460]:
print(label_vocab.lookup_tokens(range(77)))

['card_payment_fee_charged', 'direct_debit_payment_not_recognised', 'balance_not_updated_after_cheque_or_cash_deposit', 'wrong_amount_of_cash_received', 'cash_withdrawal_charge', 'transaction_charged_twice', 'declined_cash_withdrawal', 'transfer_fee_charged', 'balance_not_updated_after_bank_transfer', 'transfer_not_received_by_recipient', 'request_refund', 'card_payment_not_recognised', 'card_payment_wrong_exchange_rate', 'extra_charge_on_statement', 'wrong_exchange_rate_for_cash_withdrawal', 'Refund_not_showing_up', 'reverted_card_payment?', 'cash_withdrawal_not_recognised', 'activate_my_card', 'pending_card_payment', 'cancel_transfer', 'beneficiary_not_allowed', 'card_arrival', 'declined_card_payment', 'pending_top_up', 'pending_transfer', 'top_up_reverted', 'top_up_failed', 'pending_cash_withdrawal', 'card_linking', 'failed_transfer', 'visa_or_mastercard', 'declined_transfer', 'card_about_to_expire', 'country_support', 'getting_spare_card', 'supported_cards_and_currencies', 'transfe

In [461]:
import torch
train_x = [torch.tensor([text_vocab[word] for word in sentence]) for sentence in train_csv['text']]
test_x = [torch.tensor([text_vocab[word] for word in sentence]) for sentence in test_csv['text']]
train_y = [torch.tensor(label_vocab[word]) for word in train_csv['category']]

In [462]:
import torch
from torch.nn.utils.rnn import pad_sequence
pad_seq = pad_sequence(train_x, padding_value=PAD_IDX, batch_first=True)
print(pad_seq[0])
print(f'Pad sequence shape: {pad_seq.shape}')
test_data = pad_sequence(test_x, padding_value=PAD_IDX, batch_first=True)

tensor([ 24, 137,   2,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
          1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
          1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
          1,   1,   1])
Pad sequence shape: torch.Size([10003, 45])


In [463]:
from sklearn.model_selection import train_test_split
train_data, val_data, train_labels, val_labels = train_test_split(pad_seq, train_y, train_size=0.8, shuffle=True, random_state=123)
print(f'Train data: {len(train_data)}')
print(f'Validation data: {len(val_data)}')

Train data: 8002
Validation data: 2001


In [464]:
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
class TextDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return len(self.x)

train_dataset = TextDataset(train_data, train_labels)
val_dataset = TextDataset(val_data, val_labels)
test_dataset = TextDataset(test_data, torch.zeros(test_data.shape[0]))  # No labels for test data



batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1)

In [465]:
# Get a batch of data
x_batch, y_batch = next(iter(train_loader))

# Print the shape of the batch
print(f'x_batch shape: {x_batch.shape}')
print(f'y_batch shape: {y_batch.shape}')

x_batch shape: torch.Size([32, 45])
y_batch shape: torch.Size([32])


# How do you choose the tokenizer for this task? Could we use the white space to tokenize the text? What about using the complicated tokenizer instead? Make some discussion.(5%)
1. 關於tokenizer的部分，因為這次的資料集大多是簡單的英文句子，並沒有複雜及特殊的文體，所以我在這次作業是使用空格來分割單字。
2. 當然可以使用空格來做tokenize理由如前述，但是必須先把句子進行處理，例如轉成小寫、去除標點符號，這樣才能夠正確的分割單字。
3. 如果使用複雜的tokenizer，例如nltk或spacy，可能會有更好的效果，但是在這次的資料集中，使用空格來分割單字已經可以得到不錯的結果，所以我選擇使用空格來分割單字。

# Why we need the special tokens like ⟨pad⟩, ⟨unk⟩? (2%)
⟨pad⟩: 在訓練模型時，因為每筆資料裡的句子長度不一定相同，所以需要將每個句子的長度統一，這時候就需要使用padding token來做padding。
⟨unk⟩: 在訓練模型時，如果我們把所有的單字都收進字典會太耗資源，所以會挑一些出現次數高的並不會全部收錄，或者是測試資料中涵蓋了訓練資料裡沒有的單字，為了避免模型無法辨識，這時候就需要使用unkown token來代表這些不在字典裡的單字。

# Briefly explain how your procedure is run to handle the text data. (3%)
首先將資料集中的句子轉成小寫，並去除<'>外的標點符號，接著將句子使用空格分割成單字，再去除了英文中的stop words，接著建立對應的字典，將每個單字轉成對應的index，最後將每個句子padding到相同的長度，並將資料集分成訓練集和驗證集。

In [466]:
import torch.nn as nn
import math


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

class TransformerModel(nn.Module):
    def __init__(self, vocab_size,
                 emb_size,
                 nhead,
                 n_layers,
                 n_hidden,
                 n_classes,
                 pretrained_embeddings,
                 dropout):
        super(TransformerModel, self).__init__()
        # self.embedding = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=True)
        self.embedding = nn.Embedding.from_pretrained(pretrained_embeddings)
        self.pos_encoder = PositionalEncoding(emb_size, dropout)
        self.transformer = nn.Transformer(d_model=emb_size, nhead=nhead, num_encoder_layers=n_layers, num_decoder_layers=n_layers, dim_feedforward=n_hidden, dropout=dropout, batch_first=True)
        self.linear = nn.Linear(emb_size, n_classes)

    def forward(self, src, tgt):
        src = self.embedding(src)
        src = self.pos_encoder(src)
        tgt = self.embedding(tgt)
        tgt = self.pos_encoder(tgt)
        output = self.transformer(src, tgt)
        output = output.mean(dim=1)
        output = self.linear(output)
        return output
        

In [467]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [468]:
from torchtext.vocab import GloVe

vocab_size = INPUT_DIM
emb_size = 200
nhead = 8
n_layers = 2
n_hidden = 50
n_classes = OUTPUT_DIM
dropout = 0.3

glove = GloVe(name='6B', dim=emb_size)
pretrained_embeddings = torch.zeros(vocab_size, emb_size)
for i, token in enumerate(text_vocab.get_itos()):
    if token in glove.stoi:
        pretrained_embeddings[i] = glove.vectors[glove.stoi[token]]



In [469]:
from tqdm import tqdm

transformer_model = TransformerModel(vocab_size, emb_size, nhead, n_layers, n_hidden, n_classes, pretrained_embeddings, dropout).to(device)

lr = 0.0001 # learning rate
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(transformer_model.parameters(), lr=lr,weight_decay=1e-5)
best_val_loss = float('inf')
epochs = 100

for epoch in range(epochs):
    # Training
    transformer_model.train()
    train_loss = 0
    train_correct = 0
    tarin_bar = tqdm(train_loader, position=0, leave=True)
    for x, y in tarin_bar:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        y_pred = transformer_model(x, x)
        loss = criterion(y_pred, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        class_pred = y_pred.argmax(dim=1)
        train_correct += class_pred.eq(y).sum().item()
    train_accuracy = train_correct / len(train_dataset)

    # Validation
    transformer_model.eval()
    val_loss = 0
    val_correct = 0
    val_bar = tqdm(val_loader, position=0, leave=True)
    with torch.no_grad():
        for x, y in val_bar:
            x, y = x.to(device), y.to(device)
            y_pred = transformer_model(x, x)
            loss = criterion(y_pred, y)
            val_loss += loss.item()
            class_pred = y_pred.argmax(dim=1)
            val_correct += class_pred.eq(y).sum().item()
    val_accuracy = val_correct / len(val_dataset)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(transformer_model.state_dict(), 'model.pth')
    print(f'Epoch {epoch+1}/{epochs}, Train Loss: {train_loss/len(train_loader):.6f}, Train Acc: {train_accuracy:.6f}, Val Loss: {val_loss/len(val_loader):.6f}, Val Acc: {val_accuracy:.6f}')
  

100%|██████████| 63/63 [00:00<00:00, 466.76it/s]


Epoch 1/100, Train Loss: 4.344355, Train Acc: 0.016371, Val Loss: 4.341164, Val Acc: 0.016492


100%|██████████| 63/63 [00:00<00:00, 465.06it/s]


Epoch 2/100, Train Loss: 4.323396, Train Acc: 0.016246, Val Loss: 4.299654, Val Acc: 0.037981


100%|██████████| 63/63 [00:00<00:00, 428.80it/s]


...

Epoch 98/100, Train Loss: 0.378683, Train Acc: 0.877656, Val Loss: 0.560002, Val Acc: 0.865067


100%|██████████| 63/63 [00:00<00:00, 447.72it/s]


Epoch 99/100, Train Loss: 0.359279, Train Acc: 0.882404, Val Loss: 0.564417, Val Acc: 0.857071


100%|██████████| 63/63 [00:00<00:00, 462.43it/s]

Epoch 100/100, Train Loss: 0.360901, Train Acc: 0.881655, Val Loss: 0.579974, Val Acc: 0.856572


In [470]:
# Load the model
model = TransformerModel(vocab_size, emb_size, nhead, n_layers, n_hidden, n_classes, pretrained_embeddings, dropout).to(device)
model.load_state_dict(torch.load('model.pth'))

<All keys matched successfully>

In [471]:
from tqdm import tqdm
test_predictions = []
model.eval()

test_bar = tqdm(test_loader, position=0, leave=True)

with torch.no_grad():
    for x, _ in test_bar:
        x = x.to(device)
        y_pred = model(x, x)
        test_predictions.append(y_pred.argmax(dim=1).cpu().numpy())
print(test_predictions)

100%|██████████| 3080/3080 [00:05<00:00, 523.35it/s]

[array([36], dtype=int64), array([47], dtype=int64), array([56], dtype=int64), array([54], dtype=int64), array([4], dtype=int64), array([34], dtype=int64), array([21], dtype=int64), array([8], dtype=int64), array([15], dtype=int64), array([70], dtype=int64), array([21], dtype=int64), array([63], dtype=int64), array([5], dtype=int64), array([3], dtype=int64), array([38], dtype=int64), array([25], dtype=int64), array([67], dtype=int64), array([18], dtype=int64), array([50], dtype=int64), array([56], dtype=int64), array([24], dtype=int64), array([55], dtype=int64), array([44], dtype=int64), array([68], dtype=int64), array([39], dtype=int64), array([1], dtype=int64), array([37], dtype=int64), array([45], dtype=int64), array([67], dtype=int64), array([41], dtype=int64), array([28], dtype=int64), array([57], dtype=int64), array([34], dtype=int64), array([71], dtype=int64), array([73], dtype=int64), array([8], dtype=int64), array([26], dtype=int64), array([43], dtype=int64), array([30], dtype

In [473]:
sub_csv = pd.read_csv('submission.csv')
sub_csv['category'] = label_vocab.lookup_tokens(test_predictions)
sub_csv.to_csv('submission.csv', index=False)

# Discuss the model structure or hyperparameter setting in your design. (5%) (e.g. hyperparameters of transformer: d model, nhead, d hid , nlayers, dropout, etc. Why do you choose this setting?)
在transformer的設定裡，我是使用了pretrained的GloVe來當作embedding layer，但是一開始在train的時候，發現模型都沒有在收斂，所以我就降低了encoder和decoder還有hidden的層數，並且刻意讓embedding layer也進行訓練更新參數，使模型能更快收斂，也有設定dropout，且使用AdamW來當作optimizer，動態調整更新的參數大小，避免overfitting，至於Positional Encoding的部分則是參考torch官方的寫法。